In [1]:
pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModel

def get_bert_embeddings(sentences):
    # Load pre-trained BERT model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    model = AutoModel.from_pretrained('bert-base-uncased')

    # Tokenize sentences
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Perform mean pooling to get sentence embeddings
    attention_mask = inputs['attention_mask']
    token_embeddings = outputs[0] # First element of model_output contains all token embeddings

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    return sum_embeddings / sum_mask

def get_gpt2_embeddings(sentences):
    # Load pre-trained GPT-2 model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    model = AutoModel.from_pretrained('gpt2')

    # GPT-2 does not have a pad token by default, we set it to eos token
    tokenizer.pad_token = tokenizer.eos_token

    # Tokenize sentences
    inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        outputs = model(**inputs)

    # Perform mean pooling
    attention_mask = inputs['attention_mask']
    token_embeddings = outputs[0]

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    return sum_embeddings / sum_mask

# Example Sentences
sentences = [
    "Deep learning enables amazing natural language processing capabilities.",
    "Artificial intelligence is transforming the technology landscape.",
    "I love cooking fresh meals at home."
]

# Get embeddings
bert_embeds = get_bert_embeddings(sentences)
gpt2_embeds = get_gpt2_embeddings(sentences)

print("BERT Sentence Embeddings Shape:", bert_embeds.shape)
print("GPT-2 Sentence Embeddings Shape:", gpt2_embeds.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BERT Sentence Embeddings Shape: torch.Size([3, 768])
GPT-2 Sentence Embeddings Shape: torch.Size([3, 768])


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute and compare similarity matrices
bert_sim = cosine_similarity(bert_embeds.numpy())
gpt2_sim = cosine_similarity(gpt2_embeds.numpy())

print("BERT Cosine Similarity Matrix:")
print(bert_sim)

print("\nGPT-2 Cosine Similarity Matrix:")
print(gpt2_sim)

BERT Cosine Similarity Matrix:
[[0.99999976 0.66408247 0.55754673]
 [0.66408247 1.         0.5850641 ]
 [0.55754673 0.5850641  1.        ]]

GPT-2 Cosine Similarity Matrix:
[[1.0000002  0.99820065 0.99679583]
 [0.99820065 1.0000004  0.9965939 ]
 [0.99679583 0.9965939  1.0000001 ]]
